# Preprocessing pipeline
This is the entire preprocessing, it should be done in this order aswell. We are constantly referencing "the paper" which is to be understood as this paper: https://onlinelibrary.wiley.com/doi/10.1155/2014/781670


## Setup
Read the dataset and define our target

In [45]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
df = pd.read_csv("../data/splits/train.csv")
TARGET = "readmitted"

## Step 1: Remove exact duplicate rows
Actually there are none in this dataset but whatever.

In [46]:
df = df.drop_duplicates()

## Step 2: Keep only the first encounter per patient
Thise is an idea directly stolen from the paper; they explain how keeping multiple encounters from the same patient violates the independence assumption of most classifiers and causes data leakage between train/test

In [47]:
df.sort_values('encounter_id', inplace=True)
df.drop_duplicates(subset='patient_nbr', keep='first', inplace=True)

## Step 3: Remove encounters that ended in death or discharge to hospice
Another idea directly stolen from the paper; they explain how a patient who died cannot be readmitted,
so including them would distort the readmission. Sounds very reasonable

In [48]:
# discharge_disposition_id values per IDS_mapping.csv:
#   11 = Expired
#   13 = Hospice / home
#   14 = Hospice / medical facility
#   19 = Expired at home (Medicaid/hospice)
#   20 = Expired in a medical facility (Medicaid/hospice)
#   21 = Expired, place unknown (Medicaid/hospice)
HOSPICE_DEATH_IDS = [11, 13, 14, 19, 20, 21]
df = df[~df['discharge_disposition_id'].isin(HOSPICE_DEATH_IDS)]

## Step 4: Encode the target variable as binary
This Step could be done in two different ways, either encode '<30' and '>30' as 1 and 'No' as 0, or have only '<30' as 1, and '>30' and 'No' as 0. The paper used the later; explaining how early readmission (<30 days) is the clinically critical outcome.

In [49]:
df[TARGET] = df[TARGET].map(lambda x: 1 if x == '<30' else 0)
print("Target distribution:\n", df[TARGET].value_counts(normalize=True))

Target distribution:
 readmitted
0    0.905931
1    0.094069
Name: proportion, dtype: float64


## Step 5: Drop identifier columns
These columns are just identifiers, there is no predictive value for them.

In [50]:
IDENTIFIER_COLS = ["id", "encounter_id", "patient_nbr"]
df.drop(columns=IDENTIFIER_COLS, inplace=True)

## Step 6: Replace '?' sentinel with proper NaN
Several numerical columns use '?' to denote missing values instead of a NaN.

In [51]:
df.replace('?', pd.NA, inplace=True)

## Step 7: Drop columns with >10% missing data (except those with clinical justification)
A feature with >10% missing values risks introducing systematic bias during
imputation. We drop weight (97% missing) altough this probably clinically relevant there is simply not enough data, if i remember correctly it was not required to take records of weights before 2009.
The payer_code payer_code (52% missing) was also removed since it has no clinical relevance.
*Exception*: medical_specialty, A1Cresult, and max_glu_serum are excluded from this drop see Step 8.

In [52]:
# weight (97% missing)
# payer_code (52% missing)
MANUALLY_DROPPED = ['weight', 'payer_code']
df.drop(columns=MANUALLY_DROPPED, inplace=True)

## Step 8: Handle A1Cresult, max_glu_serum, and medical_specialty
These columns are ~83%, ~95%, and ~50% missing respectively, which would normally trigger removal. However, the paper's central finding is that the actually meassuring HbA1c is itself a significant predictor of readmission
Missingness is therefore informative.
We encode "not tested" as its own category rather than discarding the feature.

The paper also explicitly kept medical_specialty explaining; how a physician's specialty is a meaningful clinical signal for how diabetes is managed. 

In [53]:
df['A1Cresult']     = df['A1Cresult'].fillna('NotTested')
df['max_glu_serum'] = df['max_glu_serum'].fillna('NotTested')
df['medical_specialty'] = df['medical_specialty'].fillna('Unknown')

## Step 9: Drop zero-variance columns
Columns where every row has the same value contribute zero information.
In this dataset, 'examide' and 'citoglipton' are always 'No'.

In [54]:
zero_variance_cols = [
    col for col in df.columns
    if col != TARGET and df[col].nunique() <= 1
]
print(f"Zero-variance columns dropped: {zero_variance_cols}")
df.drop(columns=zero_variance_cols, inplace=True)

Zero-variance columns dropped: ['troglitazone', 'examide', 'citoglipton', 'glimepiride-pioglitazone']


## Step 10: Map ICD-9 diagnosis codes to clinical categories
diag_1/2/3 each have ~850-950 unique raw ICD-9 codes. The paper maps them to 9 clinically meaningful
groups, which reduces dimensionality.

In [55]:
def map_icd9_to_category(code):
    if pd.isna(code):
        return 'Other'
    try:
        c = float(code)
    except ValueError:
        return 'Other'  # E-codes, V-codes → Other
    if 390 <= c <= 459 or c == 785: return 'Circulatory'
    if 460 <= c <= 519 or c == 786: return 'Respiratory'
    if 520 <= c <= 579 or c == 787: return 'Digestive'
    if 250 <= c < 251:              return 'Diabetes'
    if 800 <= c <= 999:             return 'Injury'
    if 710 <= c <= 739:             return 'Musculoskeletal'
    if 580 <= c <= 629 or c == 788: return 'Genitourinary'
    if 140 <= c <= 239:             return 'Neoplasms'
    return 'Other'

for col in ['diag_1', 'diag_2', 'diag_3']:
    df[col] = df[col].apply(map_icd9_to_category)

## Step 11: Encode age as 3 ordered groups
Idea directly stolen from the paper, instead of assigning a numerical value to each age span, they encoded ages as three groups, young, medium, senior.

In [56]:
def map_age_to_group(age_bracket):
    young = ['[0-10)', '[10-20)', '[20-30)']
    middle = ['[30-40)', '[40-50)', '[50-60)']
    if age_bracket in young:   return 'Young'       # [0, 30)
    if age_bracket in middle:  return 'Middle'      # [30, 60)
    return 'Senior'                                 # [60, 100+)

df['age'] = df['age'].map(map_age_to_group)

## Step 12: Encode medication columns and engineer aggregate features
All 23 medication columns use values No/Steady/Up/Down. We encode them ordinally (0=not prescribed, 1=steady, 2=down, 3=up) to preserve the direction of dosage change, which is clinically relevant acording to the paper

In [57]:
MED_COLS = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
    'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide', 'insulin',
    'glyburide-metformin', 'glipizide-metformin',
    'glimepiride-pioglitazone', 'metformin-rosiglitazone',
    'metformin-pioglitazone'
]

# Zero-variance are already dropped)
MED_COLS = [c for c in MED_COLS if c in df.columns]
MED_MAP = {'No': 0, 'Steady': 1, 'Down': 2, 'Up': 3}

for col in MED_COLS:
    df[col] = df[col].map(MED_MAP)

## Step 13: Handle missing values in race
After all steps above, only 'race' retains a small number of NaNs (~2%). The paper treats 'Missing' race as its own category, noting it actually had the lowest readmission rate (7.5%).

In [58]:
df['race'] = df['race'].fillna('Unknown')

## Step 14: Train / test split
This must happen before doing things like one-hot encoding or scaling numerical values for logistic regression.

In [59]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

Train size: (42368, 41), Test size: (10593, 41)


## Step 15: One hot encoding

In [60]:
CATEGORICAL_COLS = [
    'race', 'gender', 'age', 'change', 'diabetesMed',
    'diag_1', 'diag_2', 'diag_3',
    'A1Cresult', 'max_glu_serum', 'medical_specialty',
    'admission_type_id', 'discharge_disposition_id', 'admission_source_id'
]

# Maybe a column has are already dropped
CATEGORICAL_COLS = [c for c in CATEGORICAL_COLS if c in X_train.columns]

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(X_train[CATEGORICAL_COLS])  # fit on train only

train_cat = pd.DataFrame(
    ohe.transform(X_train[CATEGORICAL_COLS]),
    columns=ohe.get_feature_names_out(CATEGORICAL_COLS),
    index=X_train.index
)
test_cat = pd.DataFrame(
    ohe.transform(X_test[CATEGORICAL_COLS]),
    columns=ohe.get_feature_names_out(CATEGORICAL_COLS),
    index=X_test.index
)

X_train = pd.concat([X_train.drop(columns=CATEGORICAL_COLS), train_cat], axis=1)
X_test  = pd.concat([X_test.drop(columns=CATEGORICAL_COLS),  test_cat],  axis=1)

## Step 16: Scaling

In [61]:
NUMERICAL_COLS = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses'
]

# Maybe a column has are already dropped
NUMERICAL_COLS = [c for c in NUMERICAL_COLS if c in X_train.columns]

scaler = StandardScaler()
scaler.fit(X_train[NUMERICAL_COLS])  # fit on train only
X_train[NUMERICAL_COLS] = scaler.transform(X_train[NUMERICAL_COLS])
X_test[NUMERICAL_COLS]  = scaler.transform(X_test[NUMERICAL_COLS])

## Done, final printing

In [62]:
print(f"Final feature count: {X_train.shape[1]}")
print("Preprocessing complete.")

Final feature count: 191
Preprocessing complete.
